# 03 — Model Comparison

> **Educational research only.** Synthetic data, simulated results, not financial advice.

Walk-forward comparison with Newey-West IC inference and PBO.

In [ ]:
import pandas as pd
from alphaforge.data import SyntheticMarketConfig, generate_synthetic_market
from alphaforge.features import build_features
from alphaforge.labels import build_labels
from alphaforge.training import run_walk_forward

panel = generate_synthetic_market(SyntheticMarketConfig(n_symbols=10, n_days=1500, seed=42))
features = build_features(panel, benchmark_symbol='BENCH')
labels = build_labels(panel, benchmark_symbol='BENCH', horizons=[1, 5, 20])
specs = [
    {'name': 'zero_baseline'},
    {'name': 'momentum_baseline', 'params': {'feature': 'momentum_20', 'scale': 0.05}},
    {'name': 'ridge', 'params': {'alpha': 10.0}},
    {'name': 'gradient_boosting', 'params': {'max_iter': 150, 'max_depth': 4}},
]
wf = {'scheme': 'expanding', 'min_train_days': 504, 'test_days': 126, 'step_days': 126, 'embargo_days': 21}
result = run_walk_forward(features, labels, specs, target='fwd_ret_5', config=wf, max_horizon=20)
result.metrics.groupby('model')[['rank_ic', 'directional_accuracy']].mean()

In [ ]:
from alphaforge.evaluation import information_coefficient_by_date, ic_summary
rows = []
for name, g in result.predictions.groupby('model'):
    rows.append({'model': name, **ic_summary(information_coefficient_by_date(g))})
pd.DataFrame(rows).sort_values('mean_ic', ascending=False)

In [ ]:
from alphaforge.evaluation import probability_of_backtest_overfitting
ic_matrix = pd.DataFrame({
    name: information_coefficient_by_date(g).set_index('date')['rank_ic']
    for name, g in result.predictions.groupby('model')
})
pbo = probability_of_backtest_overfitting(ic_matrix, n_blocks=8)
{k: v for k, v in pbo.items() if k != 'logits'}

A PBO near 0.5 means the in-sample winner is likely selection bias; a low PBO means the ranking is stable across IS/OOS partitions.